In [1]:
import os, pandas as pd, numpy as np

for folder in ["bronze", "silver", "gold"]:
    os.makedirs(folder, exist_ok=True)

In [2]:
!wget -q -O book-crossings.zip https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
!unzip -qo book-crossings.zip

In [3]:
users = pd.read_csv("BX-Users.csv", sep=";", encoding="latin-1", escapechar="\\", on_bad_lines="skip")
books = pd.read_csv("BX-Books.csv", sep=";", encoding="latin-1", escapechar="\\", on_bad_lines="skip", dtype=str)
ratings = pd.read_csv("BX-Book-Ratings.csv", sep=";", encoding="latin-1", escapechar="\\", on_bad_lines="skip")

In [4]:
now = pd.Timestamp.now()
users["ingestion_timestamp"] = now
books["ingestion_timestamp"] = now
ratings["ingestion_timestamp"] = now

In [5]:
users.to_parquet("bronze/users.parquet", index=False)
books.to_parquet("bronze/books.parquet", index=False)
ratings.to_parquet("bronze/ratings.parquet", index=False)

In [6]:
users = pd.read_parquet("bronze/users.parquet")
books = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")

In [7]:
for name, df in [("Users", users), ("Books", books), ("Ratings", ratings)]:
    print(name, df.shape)
    print(df.dtypes)
    print("Missing:\n", df.isna().sum(), "\n")

Users (278858, 4)
User-ID                         int64
Location                       object
Age                           float64
ingestion_timestamp    datetime64[us]
dtype: object
Missing:
 User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64 

Books (271379, 9)
ISBN                           object
Book-Title                     object
Book-Author                    object
Year-Of-Publication            object
Publisher                      object
Image-URL-S                    object
Image-URL-M                    object
Image-URL-L                    object
ingestion_timestamp    datetime64[us]
dtype: object
Missing:
 ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            0
ingestion_timestamp    0
dtype: int64 

Ratings (1149780, 4)
User-ID                   

In [8]:
print(ratings["Book-Rating"].value_counts().sort_index())
print("Zero ratings %:", (ratings["Book-Rating"] == 0).mean() * 100)

Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64
Zero ratings %: 62.28226269373272


In [9]:
ages = pd.to_numeric(users["Age"], errors="coerce")
print("Bad ages (<5 or >100):", ((ages < 5) | (ages > 100)).sum())

years = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
print("Bad years (0 or >2026):", ((years == 0) | (years > 2026)).sum())

Bad ages (<5 or >100): 1248
Bad years (0 or >2026): 4630


In [10]:
print("Ratings per user:\n", ratings["User-ID"].value_counts().describe())
print("Ratings per book:\n", ratings["ISBN"].value_counts().describe())

Ratings per user:
 count    105283.000000
mean         10.920851
std          90.562825
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max       13602.000000
Name: count, dtype: float64
Ratings per book:
 count    340556.000000
mean          3.376185
std          12.436252
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        2502.000000
Name: count, dtype: float64


In [11]:
total_pairs = ratings["User-ID"].nunique() * ratings["ISBN"].nunique()
filled_pct = (len(ratings) / total_pairs) * 100
print(f"Filled: {filled_pct:.6f}% | Empty: {100 - filled_pct:.6f}%")

Filled: 0.003207% | Empty: 99.996793%


In [12]:
books = books.drop(columns=["Image-URL-S", "Image-URL-M", "Image-URL-L"])
books["ISBN"] = books["ISBN"].str.strip().str.upper()
ratings["ISBN"] = ratings["ISBN"].str.strip().str.upper()
books["Book-Title"] = books["Book-Title"].str.strip()
books["Book-Author"] = books["Book-Author"].fillna("Unknown").str.strip().str.title()
books["Publisher"] = books["Publisher"].fillna("Unknown").str.strip().str.title()

In [13]:
users = users.drop_duplicates(subset=["User-ID"])
books = books.drop_duplicates(subset=["ISBN"])
ratings = ratings.drop_duplicates(subset=["User-ID", "ISBN"])

In [14]:
ratings = ratings[ratings["Book-Rating"].between(1, 10)]

In [15]:
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
users.loc[(users["Age"] < 5) | (users["Age"] > 100), "Age"] = np.nan
users["Age"] = users["Age"].fillna(users["Age"].median())

In [16]:
loc = users["Location"].fillna("").str.split(",", n=2, expand=True)
users["city"] = loc[0].str.strip().str.lower()
users["state"] = loc[1].fillna("").str.strip().str.lower()
users["country"] = loc[2].fillna("").str.strip().str.lower()
users = users.drop(columns=["Location"])

In [17]:
books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
books.loc[(books["Year-Of-Publication"] <= 0) | (books["Year-Of-Publication"] > 2026), "Year-Of-Publication"] = np.nan

In [18]:
ratings = ratings[
    ratings["User-ID"].isin(users["User-ID"]) &
    ratings["ISBN"].isin(books["ISBN"])
]

In [19]:
for _ in range(5):
    u_ok = ratings["User-ID"].value_counts()
    b_ok = ratings["ISBN"].value_counts()
    ratings = ratings[
        ratings["User-ID"].isin(u_ok[u_ok >= 10].index) &
        ratings["ISBN"].isin(b_ok[b_ok >= 10].index)
    ]

In [20]:
users.to_parquet("silver/users.parquet", index=False)
books.to_parquet("silver/books.parquet", index=False)
ratings.to_parquet("silver/ratings.parquet", index=False)

In [21]:
from scipy.sparse import csr_matrix, save_npz

books = pd.read_parquet("silver/books.parquet")
ratings = pd.read_parquet("silver/ratings.parquet")

user_ids = ratings["User-ID"].unique()
book_isbns = ratings["ISBN"].unique()

user_map = {u: i for i, u in enumerate(user_ids)}
book_map = {b: i for i, b in enumerate(book_isbns)}

In [22]:
rows = ratings["User-ID"].map(user_map)
cols = ratings["ISBN"].map(book_map)
vals = ratings["Book-Rating"].astype(float)

matrix = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(book_isbns)))
save_npz("gold/user_book_matrix.npz", matrix)

In [23]:
book_lookup = pd.DataFrame({"book_idx": range(len(book_isbns)), "ISBN": book_isbns})
book_lookup = book_lookup.merge(books[["ISBN", "Book-Title", "Book-Author"]], on="ISBN", how="left")
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

user_lookup = pd.DataFrame({"user_idx": range(len(user_ids)), "User-ID": user_ids})
user_lookup.to_parquet("gold/user_lookup.parquet", index=False)

In [24]:
user_stats = ratings.groupby("User-ID")["Book-Rating"].agg(avg_rating="mean", rating_count="count").reset_index()
book_stats = ratings.groupby("ISBN")["Book-Rating"].agg(avg_rating="mean", rating_count="count").reset_index()

user_stats.to_parquet("gold/user_aggregates.parquet", index=False)
book_stats.to_parquet("gold/book_aggregates.parquet", index=False)

In [25]:
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

user_book = load_npz("gold/user_book_matrix.npz")
book_user = user_book.T.tocsr()
book_lookup = pd.read_parquet("gold/book_lookup.parquet")
user_lookup = pd.read_parquet("gold/user_lookup.parquet")

In [26]:
model = NearestNeighbors(metric="cosine", algorithm="brute")
model.fit(book_user)

NearestNeighbors(algorithm='brute', metric='cosine')

In [27]:
def similar_books(title, n=10):
    match = book_lookup[book_lookup["Book-Title"].str.contains(title, case=False, na=False)]
    idx = match.iloc[0]["book_idx"]
    dists, nbrs = model.kneighbors(book_user[idx], n_neighbors=n + 1)
    result = book_lookup.iloc[nbrs[0][1:]][["Book-Title", "Book-Author"]].copy()
    result["Similarity"] = (1 - dists[0][1:]).round(3)
    return result

In [28]:
def recommend(user_id, n=10):
    u_idx = user_lookup.loc[user_lookup["User-ID"] == user_id, "user_idx"].iloc[0]
    row = user_book[u_idx]
    already_read = set(row.indices)
    min_rating = 7 if (row.data >= 7).any() else row.data.mean()

    scores = {}
    for b_idx, rating in zip(row.indices, row.data):
        if rating < min_rating:
            continue
        dists, nbrs = model.kneighbors(book_user[b_idx], n_neighbors=15)
        for dist, nb in zip(dists[0], nbrs[0]):
            if nb not in already_read:
                scores[nb] = scores.get(nb, 0) + (1 - dist) * rating

    top_books = sorted(scores, key=scores.get, reverse=True)[:n]
    return book_lookup.iloc[top_books][["Book-Title", "Book-Author"]]

In [29]:
print(similar_books("Harry Potter"))

                                             Book-Title      Book-Author  \
178    Harry Potter and the Chamber of Secrets (Book 2)    J. K. Rowling   
180   Harry Potter and the Prisoner of Azkaban (Book 3)    J. K. Rowling   
664        Harry Potter and the Goblet of Fire (Book 4)    J. K. Rowling   
181        Harry Potter and the Goblet of Fire (Book 4)    J. K. Rowling   
292   Harry Potter and the Order of the Phoenix (Boo...    J. K. Rowling   
179   Harry Potter and the Prisoner of Azkaban (Book 3)    J. K. Rowling   
177    Harry Potter and the Chamber of Secrets (Book 2)    J. K. Rowling   
549       Fried Green Tomatoes at the Whistle Stop Cafe     Fannie Flagg   
2042                                         Love Story      Erich Segal   
1348  Anne of Green Gables (Anne of Green Gables Nov...  L.M. Montgomery   

      Similarity  
178        0.403  
180        0.321  
664        0.260  
181        0.252  
292        0.246  
179        0.201  
177        0.187  
549        

In [30]:
sample_users = user_lookup["User-ID"].iloc[:3].tolist()

for uid in sample_users:
    u_idx = user_lookup.loc[user_lookup["User-ID"] == uid, "user_idx"].iloc[0]
    row = user_book[u_idx]

    liked = book_lookup.iloc[row.indices][["Book-Title", "Book-Author"]].copy()
    liked["Rating"] = row.data
    liked = liked.sort_values("Rating", ascending=False).head(10)

    print(f"\n=== USER {uid}: TOP RATED BOOKS ===")
    print(liked.to_string(index=False))
    print(f"\n=== USER {uid}: TOP 10 RECOMMENDATIONS ===")
    print(recommend(uid, n=10).to_string(index=False))


=== USER 277427: TOP RATED BOOKS ===
                                                                                              Book-Title        Book-Author  Rating
                                Politically Correct Bedtime Stories: Modern Tales for Our Life and Times  James Finn Garner    10.0
                                                                                         The Murder Book Jonathan Kellerman    10.0
The Venetian's Wife: A Strangely Sensual Tale of a Renaissance Explorer, a Computer, and a Metamorphosis       Nick Bantock    10.0
            The Golden Mean: In Which the Extraordinary Correspondence of Griffin &amp; Sabine Concludes       Nick Bantock    10.0
                                                                Dilbert Fugitive From The Cubicle Police        Scott Adams    10.0
                                                    Creative Companion: How to Free Your Creative Spirit               Sark    10.0
                                      

### Sanity Check Observations

- **User 277427 (Comics, Illustrated Fiction & Thrillers):** This user rated Nick Bantock (*The Golden Mean*), Jonathan Kellerman (*The Murder Book*), and comic collections by Gary Larson and Scott Adams a 10/10. The model accurately recommended the other books in Nick Bantock's *Griffin & Sabine* series (*Sabine's Notebook* and *Griffin & Sabine*), another Jonathan Kellerman novel (*Flesh and Blood*), and several *Calvin and Hobbes* collections by Bill Watterson that match their humor/comic preference.
- **User 277523 (Crime & Legal Thrillers):** This user rated thriller authors like James Patterson, Jonathan Kellerman, John Grisham, and David Baldacci 10/10. The recommendations strongly match this genre, returning unread titles by Jonathan Kellerman (*A Cold Heart*) and James Patterson (*Sam's Letters to Jennifer*) alongside closely related crime/mystery authors such as Michael Connelly, Patricia Cornwell, and John Sandford.
- **User 277945 (Romance & Nora Roberts):** Most of this user's highest-rated books are by Nora Roberts, including *Key of Knowledge* and *Key of Valor* from the *Key Trilogy*. The model recommended 8 more unread Nora Roberts novels—including *Key of Light* (the missing book from the same trilogy)—plus 2 books by J.D. Robb (*Purity in Death* and *Reunion in Death*), which is actually Nora Roberts' pen name.
- **Overall Validation:** Across all three users, the recommendations closely match their favorite authors, series, and genres, and none of their already-rated books appear in the suggested lists.